In [5]:
# python version 3.7.2
import numpy as np
from uncertainties import ufloat, umath

# -- 距离/视差输入区（全部按你的原始变量，单位注释保留）
d_k0 = ufloat(157, 1.2)*3.09e16    # m, timing old
d_k = ufloat(156.96, 0.11)*3.09e16 # m, timing new. reardon:156.96, 0.11 (Reardon et al. 2024 ApJL 971, L18); ATNF PSRCAT: 156.79, 0.25
pi_k = 1/d_k*3.09e16*1000          # mas, timing new
d_pi0 = ufloat(156.3, 1.3)*3.09e16 # m, Deller
#pi_v = ufloat(6.434053, 0.02294112309)        # mas, our new
pi_v = ufloat(6.43, 0.04)        # mas, our new
d_pi = 1/pi_v*3.09e19              # m, our new

# -- proper motion输入区
pm_x = ufloat(121.4429, 0.050)/1000/3600/180*np.pi # rad/yr    chen_old:121.453, 0.052  ding:121.527273 + 0.019033; ATNF PSRCAT reardon: 121.4429(5)
pm_y = ufloat(-71.4715, 0.050)/1000/3600/180*np.pi # rad/yr    chen_old:-71.457, 0.086  ding:-71.546800 + 0.014013; ATNF PSRCAT reardon: −71.4715(5)
mu = umath.sqrt((pm_x)**2+(pm_y)**2)              # rad/yr

# -- c的定义与单位说明
c = 3e8/(1/3600/24/365)  # m/yr

# -- 输出区
print("pi_v (our new, mas):", pi_v)
print("d_pi (our new, pc):", d_pi/3.09e19*1000)
print("pi_k (timing new, mas):", pi_k)
print("d_pi (our new, m):", d_pi)
print("d_k (timing new, m):", d_k)
print("mu (rad/yr):", mu)

# -- 公式输出
dG_over_G_Deller = (mu**2)/(2*c)*(d_pi0-d_k0)
dG_over_G_ournew = (mu**2)/(2*c)*(d_pi-d_k)

print("\ndotG/G (Deller, 1/s):", dG_over_G_Deller)
print("dotG/G (our new, 1/s):", dG_over_G_ournew)


pi_v (our new, mas): 6.43+/-0.04
d_pi (our new, pc): 155.5+/-1.0
pi_k (timing new, mas): 6.371+/-0.004
d_pi (our new, m): (4.806+/-0.030)e+18
d_k (timing new, m): (4.8501+/-0.0034)e+18
mu (rad/yr): (6.8317+/-0.0024)e-07

dotG/G (Deller, 1/s): (-0.5+/-1.3)e-12
dotG/G (our new, 1/s): (-1.1+/-0.7)e-12


In [1]:
# ============================================================
# Recompute dotG/G for PSR J0437-4715
# using Reardon et al. (2024) timing parameters
# and our VLBI parallax distance D_pi.
#
# Formula:
#   dotG/G = -1/(2 Pb) * [
#       Pbdot_obs - Pbdot_Gal - Pbdot_GW - Pbdot_shk(D_pi)
#   ]
#
# where:
#   Pbdot_shk(D_pi) = (mu^2 D_pi / c) * Pb
#
# Cross-check:
#   dotG/G = (mu^2 / 2c) * (D_pi - D_k)
# ============================================================

import numpy as np

# -----------------------------
# Physical constants
# -----------------------------
c = 299792458.0                         # speed of light, m/s
pc = 3.0856775814913673e16              # parsec, m
day = 86400.0                           # day, s
yr = 365.25 * 86400.0                   # Julian year, s
mas_to_rad = np.pi / (180.0 * 3600.0 * 1000.0)

# -----------------------------
# Reardon et al. (2024), PPTA-DR2e parameters
# Table 1 and Sec. 3.3
# -----------------------------

# Orbital period
Pb_day = 5.74104582                     # days
Pb_day_err = 0.00000016                 # days; from 5.74104582(16)
Pb = Pb_day * day                       # seconds
Pb_err = Pb_day_err * day               # seconds

# Observed orbital-period derivative
# Table 1: Pbdot = 3.7329(15) x 10^-12
Pbdot_obs = 3.7329e-12
Pbdot_obs_err = 0.0015e-12

# Galactic acceleration correction
# Sec. 3.3: Pbdot_Gal = (-2.24 +/- 0.20) x 10^-14
Pbdot_Gal = -2.24e-14
Pbdot_Gal_err = 0.20e-14

# Gravitational-wave damping contribution
# Sec. 3.3: Pbdot_GW ~ -3.2 x 10^-16
# No uncertainty is quoted in the paper; set to zero here by default.
Pbdot_GW = -3.2e-16
Pbdot_GW_err = 0.0

# Proper motion components
# Table 1:
# mu_alpha cos(delta) = 121.4429(5) mas/yr
# mu_delta            = -71.4715(5) mas/yr
mu_alpha = 121.4429                    # mas/yr
mu_alpha_err = 0.0005                  # mas/yr
mu_delta = -71.4715                    # mas/yr
mu_delta_err = 0.0005                  # mas/yr

# Total proper motion
mu_masyr = np.sqrt(mu_alpha**2 + mu_delta**2)
mu_rad_s = mu_masyr * mas_to_rad / yr

# -----------------------------
# Our VLBI parallax distance
# -----------------------------
Dpi_pc = 155.4                         # pc
Dpi_pc_err = 0.6                       # pc
Dpi_m = Dpi_pc * pc
Dpi_m_err = Dpi_pc_err * pc

# -----------------------------
# Reardon et al. (2024) Shklovskii distance
# for cross-check
# Table 1: D_shk = 156.96(11) pc
# -----------------------------
Dk_pc = 156.96                         # pc
Dk_pc_err = 0.11                       # pc
Dk_m = Dk_pc * pc
Dk_m_err = Dk_pc_err * pc

# ============================================================
# 1. Direct computation from Pbdot budget
# ============================================================

Pbdot_shk_Dpi = (mu_rad_s**2 * Dpi_m / c) * Pb

residual = (
    Pbdot_obs
    - Pbdot_Gal
    - Pbdot_GW
    - Pbdot_shk_Dpi
)

dotG_over_G_s = -residual / (2.0 * Pb)       # s^-1
dotG_over_G_yr = dotG_over_G_s * yr          # yr^-1

print("=== Direct Pbdot-budget calculation ===")
print(f"mu_total = {mu_masyr:.6f} mas/yr")
print(f"Pbdot_shk(D_pi) = {Pbdot_shk_Dpi:.6e}")
print(f"Residual bracket = {residual:.6e}")
print(f"dotG/G = {dotG_over_G_yr:.6e} yr^-1")
print(f"dotG/G = {dotG_over_G_yr/1e-13:.3f} x 10^-13 yr^-1")

# ============================================================
# 2. Cross-check using dotG/G = mu^2/(2c) * (D_pi - D_k)
# ============================================================

dotG_over_G_s_check = (mu_rad_s**2 / (2.0 * c)) * (Dpi_m - Dk_m)
dotG_over_G_yr_check = dotG_over_G_s_check * yr

print("\n=== Distance-difference cross-check ===")
print(f"D_pi - D_k = {Dpi_pc - Dk_pc:.3f} pc")
print(f"dotG/G = {dotG_over_G_yr_check:.6e} yr^-1")
print(f"dotG/G = {dotG_over_G_yr_check/1e-13:.3f} x 10^-13 yr^-1")

print("\nDifference between two methods:")
print(f"{(dotG_over_G_yr - dotG_over_G_yr_check):.3e} yr^-1")

=== Direct Pbdot-budget calculation ===
mu_total = 140.913283 mas/yr
Pbdot_shk(D_pi) = 3.718187e-12
Residual bracket = 3.743258e-14
dotG/G = -1.190746e-12 yr^-1
dotG/G = -11.907 x 10^-13 yr^-1

=== Distance-difference cross-check ===
D_pi - D_k = -1.560 pc
dotG/G = -1.187337e-12 yr^-1
dotG/G = -11.873 x 10^-13 yr^-1

Difference between two methods:
-3.408e-15 yr^-1


In [2]:
# ============================================================
# Monte Carlo uncertainty propagation
# Assumption: independent Gaussian errors.
# This is an approximation because the timing covariance matrix
# is not included here.
# ============================================================

rng = np.random.default_rng(12345)
N = 1_000_000

Pb_samp = rng.normal(Pb, Pb_err, N)
Pbdot_obs_samp = rng.normal(Pbdot_obs, Pbdot_obs_err, N)
Pbdot_Gal_samp = rng.normal(Pbdot_Gal, Pbdot_Gal_err, N)
Pbdot_GW_samp = rng.normal(Pbdot_GW, Pbdot_GW_err, N)

mu_alpha_samp = rng.normal(mu_alpha, mu_alpha_err, N)
mu_delta_samp = rng.normal(mu_delta, mu_delta_err, N)
mu_samp_rad_s = np.sqrt(mu_alpha_samp**2 + mu_delta_samp**2) * mas_to_rad / yr

Dpi_samp_m = rng.normal(Dpi_m, Dpi_m_err, N)

Pbdot_shk_samp = (mu_samp_rad_s**2 * Dpi_samp_m / c) * Pb_samp

residual_samp = (
    Pbdot_obs_samp
    - Pbdot_Gal_samp
    - Pbdot_GW_samp
    - Pbdot_shk_samp
)

dotG_samp_yr = -residual_samp / (2.0 * Pb_samp) * yr

mean = np.mean(dotG_samp_yr)
std_1sigma = np.std(dotG_samp_yr, ddof=1)

# 95% central interval
low95, high95 = np.percentile(dotG_samp_yr, [2.5, 97.5])

print("=== Monte Carlo result ===")
print(f"dotG/G mean = {mean:.6e} yr^-1")
print(f"1-sigma uncertainty = {std_1sigma:.6e} yr^-1")
print(f"dotG/G = ({mean/1e-13:.3f} +/- {std_1sigma/1e-13:.3f}) x 10^-13 yr^-1  [1 sigma]")
print(f"95% interval = [{low95/1e-13:.3f}, {high95/1e-13:.3f}] x 10^-13 yr^-1")

# If you want to quote a symmetric 95% uncertainty:
print(f"symmetric 95% uncertainty ~ +/- {1.96*std_1sigma/1e-13:.3f} x 10^-13 yr^-1")

=== Monte Carlo result ===
dotG/G mean = -1.190617e-12 yr^-1
1-sigma uncertainty = 4.638680e-13 yr^-1
dotG/G = (-11.906 +/- 4.639) x 10^-13 yr^-1  [1 sigma]
95% interval = [-21.008, -2.829] x 10^-13 yr^-1
symmetric 95% uncertainty ~ +/- 9.092 x 10^-13 yr^-1


In [5]:
import numpy as np
from pathlib import Path

# ============================================================
# Recompute dotG/G for PSR J0437-4715
# for multiple Reardon et al. (2024) timing data sets:
#   PPTA-DR2e, PPTA-DR2.5, PPTA-DR3
#
# Main formula:
#
#   dotG/G = -1/(2 Pb) * [
#       Pbdot_obs - Pbdot_Gal - Pbdot_GW - Pbdot_shk(D_pi)
#   ]
#
# where:
#
#   Pbdot_shk(D_pi) = (mu^2 D_pi / c) * Pb
#
# Equivalent distance-difference form:
#
#   dotG/G = mu^2/(2c) * (D_pi - D_k)
#
# Switch:
#
#   use_effective_gal_from_Dk = True
#
#     For each data set, infer an effective Galactic correction
#     from its own published D_k:
#
#       Pbdot_Gal_eff =
#           Pbdot_obs - Pbdot_GW - (mu^2 D_k / c) Pb
#
#     This makes the direct Pbdot-budget method exactly self-consistent
#     with the distance-difference method for each data set.
#
#   use_effective_gal_from_Dk = False
#
#     Use the same Reardon et al. (2024) Galactic correction
#     Pbdot_Gal = (-2.24 +/- 0.20) x 10^-14 for all data sets.
#
# Notes:
# - PB is in days in .par files.
# - PBDOT is dimensionless, i.e. s/s.
# - PMRA and PMDEC are in mas/yr.
# - This code propagates errors with independent Gaussian MC sampling.
# - It does NOT include timing covariance because covariance/posterior
#   samples are not included in the public data package.
# ============================================================

# ============================================================
# User settings
# ============================================================

use_effective_gal_from_Dk = True

datasets = {
    "PPTA-DR2e": {
        "par_file": Path("/Users/chenwen/Documents/Writing/PSR0437/Parkes_data_sets_PSR0437/2024-07-05_Reardon_John_Daniel_62929v1//data/J0437-Timing/dr2e/J0437-4715_DR2e.par"),
        "Dk_pc": 156.96,
        "Dk_pc_err": 0.11,
    },
    "PPTA-DR2.5": {
        "par_file": Path("/Users/chenwen/Documents/Writing/PSR0437/Parkes_data_sets_PSR0437/2024-07-05_Reardon_John_Daniel_62929v1//data/J0437-Timing/dr2.5/J0437-4715.par"),
        "Dk_pc": 156.69,
        "Dk_pc_err": 0.30,
    },
    "PPTA-DR3": {
        "par_file": Path("/Users/chenwen/Documents/Writing/PSR0437/Parkes_data_sets_PSR0437/2024-07-05_Reardon_John_Daniel_62929v1//data/J0437-Timing/dr3/J0437-4715.par"),
        "Dk_pc": 156.98,
        "Dk_pc_err": 0.15,
    },
}

# Our VLBI parallax distance
Dpi_pc = 155.4
Dpi_pc_err = 0.6

# Reardon et al. (2024), Sec. 3.3
# Used directly only when use_effective_gal_from_Dk = False.
# Also used as a reference value when use_effective_gal_from_Dk = True.
Pbdot_Gal_ref = -2.24e-14
Pbdot_Gal_ref_err = 0.20e-14

# Gravitational-wave damping contribution
Pbdot_GW = -3.2e-16
Pbdot_GW_err = 0.0

# Monte Carlo settings
N = 1_000_000
seed = 20260610

# ============================================================
# Constants
# ============================================================

c = 299792458.0
pc = 3.0856775814913673e16
day = 86400.0
yr = 365.25 * 86400.0
mas_to_rad = np.pi / (180.0 * 3600.0 * 1000.0)

# ============================================================
# Helper functions
# ============================================================

def read_par_parameter(par_path, name):
    """
    Read parameter value and uncertainty from a TEMPO/TEMPO2 .par file.

    Common formats:
        PARAM value fitflag error
        PARAM value
        PARAM value error

    Returns
    -------
    value : float
    error : float or np.nan
    raw_line : str
    """
    name_upper = name.upper()

    with open(par_path, "r") as f:
        for line in f:
            raw = line.strip()

            if not raw or raw.startswith("#"):
                continue

            parts = raw.split()

            if parts[0].upper() != name_upper:
                continue

            value = float(parts[1].replace("D", "E"))
            error = np.nan

            # Usual TEMPO2 format: PARAM value fitflag error
            if len(parts) >= 4:
                try:
                    error = float(parts[3].replace("D", "E"))
                except ValueError:
                    pass

            # Less common format: PARAM value error
            elif len(parts) == 3:
                try:
                    error = float(parts[2].replace("D", "E"))
                except ValueError:
                    pass

            return value, error, raw

    raise KeyError(f"Parameter {name} not found in {par_path}")


def read_dataset_parameters(par_file):
    keys = ["PB", "PBDOT", "PMRA", "PMDEC", "PX"]
    out = {}
    for key in keys:
        value, error, raw = read_par_parameter(par_file, key)
        out[key] = {"value": value, "error": error, "raw": raw}
    return out


def compute_dotG_for_dataset(name, info, rng):
    par_file = info["par_file"]
    Dk_pc = info["Dk_pc"]
    Dk_pc_err = info["Dk_pc_err"]

    if not par_file.exists():
        raise FileNotFoundError(f"{name}: par file not found: {par_file}")

    params = read_dataset_parameters(par_file)

    # Values from .par
    Pb_day = params["PB"]["value"]
    Pb_day_err = params["PB"]["error"]

    Pbdot_obs = params["PBDOT"]["value"]
    Pbdot_obs_err = params["PBDOT"]["error"]

    mu_alpha = params["PMRA"]["value"]
    mu_alpha_err = params["PMRA"]["error"]

    mu_delta = params["PMDEC"]["value"]
    mu_delta_err = params["PMDEC"]["error"]

    # Unit conversion
    Pb = Pb_day * day
    Pb_err = Pb_day_err * day

    Dpi_m = Dpi_pc * pc
    Dpi_m_err = Dpi_pc_err * pc

    Dk_m = Dk_pc * pc
    Dk_m_err = Dk_pc_err * pc

    mu_masyr = np.sqrt(mu_alpha**2 + mu_delta**2)
    mu_rad_s = mu_masyr * mas_to_rad / yr

    # Shklovskii terms
    Pbdot_shk_Dpi = (mu_rad_s**2 * Dpi_m / c) * Pb
    Pbdot_shk_Dk = (mu_rad_s**2 * Dk_m / c) * Pb

    # Effective Galactic correction
    if use_effective_gal_from_Dk:
        Pbdot_Gal_used = Pbdot_obs - Pbdot_GW - Pbdot_shk_Dk
        Pbdot_Gal_used_err_display = np.nan
        gal_mode = "effective Gal from D_k"
    else:
        Pbdot_Gal_used = Pbdot_Gal_ref
        Pbdot_Gal_used_err_display = Pbdot_Gal_ref_err
        gal_mode = "fixed Reardon Gal"

    # Central direct Pbdot-budget result
    residual = (
        Pbdot_obs
        - Pbdot_Gal_used
        - Pbdot_GW
        - Pbdot_shk_Dpi
    )

    dotG_direct_yr = -residual / (2.0 * Pb) * yr

    # Central distance-difference cross-check
    dotG_dist_yr = (mu_rad_s**2 / (2.0 * c)) * (Dpi_m - Dk_m) * yr

    # ========================================================
    # Monte Carlo
    # ========================================================

    Pb_day_samp = rng.normal(Pb_day, Pb_day_err, N)
    Pb_samp = Pb_day_samp * day

    Pbdot_obs_samp = rng.normal(Pbdot_obs, Pbdot_obs_err, N)

    Pbdot_GW_samp = rng.normal(Pbdot_GW, Pbdot_GW_err, N)

    mu_alpha_samp = rng.normal(mu_alpha, mu_alpha_err, N)
    mu_delta_samp = rng.normal(mu_delta, mu_delta_err, N)
    mu_samp_rad_s = np.sqrt(mu_alpha_samp**2 + mu_delta_samp**2) * mas_to_rad / yr

    Dpi_samp_m = rng.normal(Dpi_m, Dpi_m_err, N)
    Dk_samp_m = rng.normal(Dk_m, Dk_m_err, N)

    Pbdot_shk_Dpi_samp = (mu_samp_rad_s**2 * Dpi_samp_m / c) * Pb_samp
    Pbdot_shk_Dk_samp = (mu_samp_rad_s**2 * Dk_samp_m / c) * Pb_samp

    if use_effective_gal_from_Dk:
        # Infer Pbdot_Gal_eff for each MC sample from that sample's D_k.
        # This guarantees self-consistency with D_k.
        Pbdot_Gal_used_samp = (
            Pbdot_obs_samp
            - Pbdot_GW_samp
            - Pbdot_shk_Dk_samp
        )
    else:
        # Use the same published Galactic correction for all data sets.
        Pbdot_Gal_used_samp = rng.normal(Pbdot_Gal_ref, Pbdot_Gal_ref_err, N)

    residual_samp = (
        Pbdot_obs_samp
        - Pbdot_Gal_used_samp
        - Pbdot_GW_samp
        - Pbdot_shk_Dpi_samp
    )

    dotG_direct_samp_yr = -residual_samp / (2.0 * Pb_samp) * yr

    # Distance-difference MC
    dotG_dist_samp_yr = (
        mu_samp_rad_s**2 / (2.0 * c) * (Dpi_samp_m - Dk_samp_m)
    ) * yr

    # Statistics
    direct_mean = np.mean(dotG_direct_samp_yr)
    direct_std = np.std(dotG_direct_samp_yr, ddof=1)
    direct_low95, direct_high95 = np.percentile(dotG_direct_samp_yr, [2.5, 97.5])

    dist_mean = np.mean(dotG_dist_samp_yr)
    dist_std = np.std(dotG_dist_samp_yr, ddof=1)
    dist_low95, dist_high95 = np.percentile(dotG_dist_samp_yr, [2.5, 97.5])

    return {
        "name": name,
        "par_file": par_file,
        "params": params,
        "gal_mode": gal_mode,
        "Pb_day": Pb_day,
        "Pb_day_err": Pb_day_err,
        "Pbdot_obs": Pbdot_obs,
        "Pbdot_obs_err": Pbdot_obs_err,
        "mu_alpha": mu_alpha,
        "mu_alpha_err": mu_alpha_err,
        "mu_delta": mu_delta,
        "mu_delta_err": mu_delta_err,
        "mu_total": mu_masyr,
        "Dk_pc": Dk_pc,
        "Dk_pc_err": Dk_pc_err,
        "Pbdot_shk_Dpi": Pbdot_shk_Dpi,
        "Pbdot_shk_Dk": Pbdot_shk_Dk,
        "Pbdot_Gal_used": Pbdot_Gal_used,
        "Pbdot_Gal_used_err_display": Pbdot_Gal_used_err_display,
        "Pbdot_Gal_minus_ref": Pbdot_Gal_used - Pbdot_Gal_ref,
        "residual": residual,
        "dotG_direct_yr": dotG_direct_yr,
        "dotG_dist_yr": dotG_dist_yr,
        "method_diff_yr": dotG_direct_yr - dotG_dist_yr,
        "direct_mean": direct_mean,
        "direct_std": direct_std,
        "direct_low95": direct_low95,
        "direct_high95": direct_high95,
        "dist_mean": dist_mean,
        "dist_std": dist_std,
        "dist_low95": dist_low95,
        "dist_high95": dist_high95,
    }


# ============================================================
# Run all datasets
# ============================================================

rng = np.random.default_rng(seed)
results = []

for name, info in datasets.items():
    results.append(compute_dotG_for_dataset(name, info, rng))

# ============================================================
# Print results
# ============================================================

print("============================================================")
print("Global adopted inputs")
print("============================================================")
print(f"use_effective_gal_from_Dk = {use_effective_gal_from_Dk}")
print(f"D_pi(VLBI)       = {Dpi_pc:.3f} +/- {Dpi_pc_err:.3f} pc")
print(f"Pbdot_Gal_ref    = {Pbdot_Gal_ref:.3e} +/- {Pbdot_Gal_ref_err:.3e}")
print(f"Pbdot_GW         = {Pbdot_GW:.3e}")
print(f"MC samples       = {N}")
print("Assumption: independent Gaussian errors; no timing covariance matrix.")
print()

for r in results:
    print("============================================================")
    print(r["name"])
    print("============================================================")
    print(f"par file: {r['par_file']}")
    print(f"Galactic correction mode: {r['gal_mode']}")
    print()
    print("Parameters read from .par:")
    print(f"PB      = {r['Pb_day']:.18f} +/- {r['Pb_day_err']:.3e} day")
    print(f"PBDOT   = {r['Pbdot_obs']:.18e} +/- {r['Pbdot_obs_err']:.3e}")
    print(f"PMRA    = {r['mu_alpha']:.12f} +/- {r['mu_alpha_err']:.3e} mas/yr")
    print(f"PMDEC   = {r['mu_delta']:.12f} +/- {r['mu_delta_err']:.3e} mas/yr")
    print(f"mu_tot  = {r['mu_total']:.12f} mas/yr")
    print(f"D_k     = {r['Dk_pc']:.3f} +/- {r['Dk_pc_err']:.3f} pc")
    print()
    print("Pbdot terms:")
    print(f"Pbdot_obs          = {r['Pbdot_obs']:.9e}")
    print(f"Pbdot_shk(D_pi)    = {r['Pbdot_shk_Dpi']:.9e}")
    print(f"Pbdot_shk(D_k)     = {r['Pbdot_shk_Dk']:.9e}")
    print(f"Pbdot_Gal_used     = {r['Pbdot_Gal_used']:.9e}")
    print(f"Pbdot_Gal_used - ref = {r['Pbdot_Gal_minus_ref']:.3e}")
    print()
    print("Central value: direct Pbdot-budget method")
    print(f"Residual bracket   = {r['residual']:.9e}")
    print(f"dotG/G             = {r['dotG_direct_yr']:.9e} yr^-1")
    print(f"dotG/G             = {r['dotG_direct_yr']/1e-13:.3f} x 10^-13 yr^-1")
    print()
    print("Central value: distance-difference cross-check")
    print(f"D_pi - D_k         = {Dpi_pc - r['Dk_pc']:.6f} pc")
    print(f"dotG/G             = {r['dotG_dist_yr']:.9e} yr^-1")
    print(f"dotG/G             = {r['dotG_dist_yr']/1e-13:.3f} x 10^-13 yr^-1")
    print(f"method diff.       = {r['method_diff_yr']:.3e} yr^-1")
    print()
    print("Monte Carlo: direct Pbdot-budget method")
    print(f"dotG/G             = ({r['direct_mean']/1e-13:.3f} +/- {r['direct_std']/1e-13:.3f}) x 10^-13 yr^-1 [1 sigma]")
    print(f"95% interval       = [{r['direct_low95']/1e-13:.3f}, {r['direct_high95']/1e-13:.3f}] x 10^-13 yr^-1")
    print(f"symmetric 95%      = +/- {1.96*r['direct_std']/1e-13:.3f} x 10^-13 yr^-1")
    print()
    print("Monte Carlo: distance-difference method")
    print(f"dotG/G             = ({r['dist_mean']/1e-13:.3f} +/- {r['dist_std']/1e-13:.3f}) x 10^-13 yr^-1 [1 sigma]")
    print(f"95% interval       = [{r['dist_low95']/1e-13:.3f}, {r['dist_high95']/1e-13:.3f}] x 10^-13 yr^-1")
    print(f"symmetric 95%      = +/- {1.96*r['dist_std']/1e-13:.3f} x 10^-13 yr^-1")
    print()

# ============================================================
# Compact summary table
# ============================================================

print("============================================================")
print("Compact summary table")
print("============================================================")
print(
    f"{'Dataset':<12s} "
    f"{'Gal mode':<22s} "
    f"{'direct center':>14s} "
    f"{'direct 95err':>14s} "
    f"{'dist center':>14s} "
    f"{'dist 95err':>14s} "
    f"{'D_k(pc)':>14s}"
)
print(
    f"{'':<12s} "
    f"{'':<22s} "
    f"{'(1e-13/yr)':>14s} "
    f"{'(1e-13/yr)':>14s} "
    f"{'(1e-13/yr)':>14s} "
    f"{'(1e-13/yr)':>14s} "
    f"{'':>14s}"
)

for r in results:
    print(
        f"{r['name']:<12s} "
        f"{r['gal_mode']:<22s} "
        f"{r['direct_mean']/1e-13:14.3f} "
        f"{1.96*r['direct_std']/1e-13:14.3f} "
        f"{r['dist_mean']/1e-13:14.3f} "
        f"{1.96*r['dist_std']/1e-13:14.3f} "
        f"{r['Dk_pc']:8.2f}+/-{r['Dk_pc_err']:<4.2f}"
    )

Global adopted inputs
use_effective_gal_from_Dk = True
D_pi(VLBI)       = 155.400 +/- 0.600 pc
Pbdot_Gal_ref    = -2.240e-14 +/- 2.000e-15
Pbdot_GW         = -3.200e-16
MC samples       = 1000000
Assumption: independent Gaussian errors; no timing covariance matrix.

PPTA-DR2e
par file: /Users/chenwen/Documents/Writing/PSR0437/Parkes_data_sets_PSR0437/2024-07-05_Reardon_John_Daniel_62929v1/data/J0437-Timing/dr2e/J0437-4715_DR2e.par
Galactic correction mode: effective Gal from D_k

Parameters read from .par:
PB      = 5.741045821777144731 +/- 1.603e-07 day
PBDOT   = 3.732925016446653466e-12 +/- 1.482e-15
PMRA    = 121.442862412013 +/- 4.948e-04 mas/yr
PMDEC   = -71.471539969891 +/- 4.813e-04 mas/yr
mu_tot  = 140.913270689778 mas/yr
D_k     = 156.960 +/- 0.110 pc

Pbdot terms:
Pbdot_obs          = 3.732925016e-12
Pbdot_shk(D_pi)    = 3.718186780e-12
Pbdot_shk(D_k)     = 3.755512208e-12
Pbdot_Gal_used     = -2.226719109e-14
Pbdot_Gal_used - ref = 1.328e-16

Central value: direct Pbdot-budg